# Module `algorithms/dynamic_runner.py`

`execute_dynamic` orchestre la resolution **dynamique** : un seul vehicule execute les sous-tournees en sequence, et entre chaque transition le simulateur met a jour les couts / statuts des aretes.

Deux strategies supportees :

- `reoptimize_at_depot=True` : a chaque retour au depot, on re-resout le probleme sur les clients restants avec la matrice courante.
- `reoptimize_at_depot=False` : on suit le plan initial jusqu'au bout (strategie *plan fige*).

`DynamicExecution` contient la trace : route effectivement parcourue, couts pas a pas, nombre de reoptimisations, temps cumule passe dans le solveur.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from cesipath import GraphGenerationConfig, GraphGenerator
from cesipath.dynamic_network import DynamicNetworkSimulator
from cesipath.algorithms import execute_dynamic, grasp


## 1. Mise en place

On construit :

1. une `GraphInstance` (statique),
2. un `DynamicNetworkSimulator` (qui fait evoluer les couts),
3. on passe tout a `execute_dynamic` avec un solveur de base (ici GRASP).


In [2]:
cfg = GraphGenerationConfig(node_count=12, seed=3)
instance = GraphGenerator(cfg).generate()
simulator = DynamicNetworkSimulator(instance, seed=0)

exec_result = execute_dynamic(
    instance,
    simulator,
    grasp,
    solver_kwargs={"max_iterations": 15, "rcl_alpha": 0.3, "seed": 0},
    reoptimize_at_depot=True,
)

print(f"plan initial        : {exec_result.planned_cost:.2f}")
print(f"cout realise        : {exec_result.realized_cost:.2f}")
print(f"nb reoptimisations  : {exec_result.reoptimizations}")
print(f"temps solveur cumule: {exec_result.solver_time:.3f} s")
print(f"nb de pas           : {exec_result.total_steps}")


plan initial        : 608.50
cout realise        : 780.98
nb reoptimisations  : 2
temps solveur cumule: 0.002 s
nb de pas           : 13


## 2. Lecture de la trace

`traveled_route` est la sequence reelle de noeuds visites (y compris les passages au depot). `step_costs` est la liste des couts dynamiques effectivement payes a chaque transition.


In [3]:
print("route parcourue :")
print(" ", exec_result.traveled_route)
print()
print("couts pas a pas :")
for i, c in enumerate(exec_result.step_costs):
    u = exec_result.traveled_route[i]
    v = exec_result.traveled_route[i + 1]
    print(f"  step {i + 1:>2} : {u} -> {v}  cout={c:.2f}")


route parcourue :
  [0, 8, 7, 11, 5, 6, 9, 1, 2, 0, 3, 4, 10, 0]

couts pas a pas :
  step  1 : 0 -> 8  cout=51.22
  step  2 : 8 -> 7  cout=79.15
  step  3 : 7 -> 11  cout=50.37
  step  4 : 11 -> 5  cout=34.96
  step  5 : 5 -> 6  cout=17.38
  step  6 : 6 -> 9  cout=41.03
  step  7 : 9 -> 1  cout=27.69
  step  8 : 1 -> 2  cout=61.62
  step  9 : 2 -> 0  cout=69.86
  step 10 : 0 -> 3  cout=42.98
  step 11 : 3 -> 4  cout=74.20
  step 12 : 4 -> 10  cout=163.96
  step 13 : 10 -> 0  cout=66.56


## 3. Strategie plan fige vs re-optimisation

On fait tourner les deux sur la meme seed du simulateur pour comparaison equitable.


In [4]:
def run(reopt: bool):
    sim = DynamicNetworkSimulator(instance, seed=0)
    return execute_dynamic(
        instance, sim, grasp,
        solver_kwargs={"max_iterations": 15, "rcl_alpha": 0.3, "seed": 0},
        reoptimize_at_depot=reopt,
    )

fixed = run(False)
reopt = run(True)

print(f"plan fige         : realise={fixed.realized_cost:.2f}  reopt={fixed.reoptimizations}  temps={fixed.solver_time:.3f}s")
print(f"re-optimisation   : realise={reopt.realized_cost:.2f}  reopt={reopt.reoptimizations}  temps={reopt.solver_time:.3f}s")
gain = 100 * (fixed.realized_cost - reopt.realized_cost) / fixed.realized_cost
print(f"gain relatif : {gain:+.1f} %  (positif = re-opt meilleur)")


plan fige         : realise=666.52  reopt=1  temps=0.002s
re-optimisation   : realise=780.98  reopt=2  temps=0.002s
gain relatif : -17.2 %  (positif = re-opt meilleur)


## 4. Convention des demandes restantes

Lors d'une re-optimisation, `_restricted_solver_input` construit un `SolverInput` avec les demandes des clients deja servis mises **a zero**. Les solveurs filtrent les clients de demande nulle, ce qui les exclut sans alterer la matrice de couts (important pour conserver l'inegalite triangulaire qui passe par Δ-TSP).


## 5. Cout planifie vs cout realise

Le `planned_cost` est le cout que l'algo a calcule sur le snapshot **initial**. Le `realized_cost` integre les evolutions des couts entre chaque pas. La difference `realized - planned` mesure l'impact de la dynamique sur la performance apparente.


In [5]:
print(f"planifie : {exec_result.planned_cost:.2f}")
print(f"realise  : {exec_result.realized_cost:.2f}")
print(f"ecart    : {exec_result.realized_cost - exec_result.planned_cost:+.2f}")


planifie : 608.50
realise  : 780.98
ecart    : +172.48


## 6. Interchangeabilite des solveurs

`execute_dynamic` accepte n'importe quelle fonction de signature `SolverInput -> VRPSolution` : GRASP, SA, tabou, GA. On peut donc comparer proprement la robustesse dynamique de chaque metaheuristique - c'est exactement le role de `dynamic_benchmark.py`.
